In [33]:
import pandas as pd

df = pd.read_parquet("ranks.parquet")
print(df)


          id  pred_10d  pred_30d release_date
0       ATOM  0.218935  0.394477   2025-06-05
1        REQ  0.717949  0.822485   2025-06-05
2        CRV  0.642998  0.727811   2025-06-05
3      MAVIA  0.108481  0.151874   2025-06-05
4       SAGA  0.098619  0.323471   2025-06-05
...      ...       ...       ...          ...
27626  HMSTR  0.584785  0.565093   2025-11-12
27627  MAVIA  0.254265  0.178603   2025-11-12
27628    JUP  0.774589  0.742570   2025-11-12
27629   USTC  0.844090  0.833158   2025-11-12
27630    NIL  0.274139  0.181935   2025-11-12

[27631 rows x 4 columns]


In [34]:
df['id'].nunique()

185

In [35]:
unique_ids = df["id"].unique()
print(unique_ids)


['ATOM' 'REQ' 'CRV' 'MAVIA' 'SAGA' 'NEAR' 'MORPHO' 'MANTA' 'MOVE' 'XAI'
 'ETC' 'DOGE' 'SOPH' 'CELO' 'MAV' 'POPCAT' 'SCR' 'COMP' 'GMT' 'SOL' 'IMX'
 'JUP' 'RUNE' 'LAUNCHCOIN' 'UMA' 'TRB' 'USTC' 'AIXBT' 'IOTA' 'VIRTUAL'
 'ALGO' 'GMX' 'ANIME' 'BCH' 'BIO' 'BSV' 'NXPC' 'MOODENG' 'TNSR' 'HBAR'
 'SNX' 'ZEREBRO' 'HYPER' 'SAND' 'BERA' 'PURR' 'GAS' 'LDO' 'ONDO' 'DYDX'
 'FTT' 'TON' 'EIGEN' 'LTC' 'BLAST' 'AI16Z' 'OMNI' 'AAVE' 'OGN' 'SUI'
 'MEME' 'FXS' 'NEIROETH' 'NIL' 'CFX' 'ME' 'XRP' 'TIA' 'BNB' 'NOT' 'IP'
 'OM' 'TAO' 'OP' 'CAKE' 'AVAX' 'kPEPE' 'GALA' 'MNT' 'BOME' 'SUPER' 'SEI'
 'VINE' 'KAS' 'BABY' 'STX' 'S' 'FARTCOIN' 'STG' 'RENDER' 'ENA' 'LINK'
 'ARB' 'ARK' 'BIGTIME' 'BTC' 'ETH' 'RSR' 'kDOGS' 'BRETT' 'BANANA' 'XLM'
 'INJ' 'ENS' 'AR' 'DOT' 'SPX' 'ETHFI' 'PAXG' 'kLUNC' 'GOAT' 'kSHIB' 'FIL'
 'MEW' 'STRK' 'TRX' 'ZK' 'KAITO' 'PENGU' 'kBONK' 'VVV' 'ORDI' 'INIT' 'APT'
 'REZ' 'LAYER' 'ZEN' 'SUSHI' 'kFLOKI' 'ADA' 'kNEIRO' 'PEOPLE' 'ZORA'
 'PENDLE' 'APE' 'HYPE' 'FET' 'CHILLGUY' 'MELANIA' 'GRIFFAIN' 'PNUT'

In [36]:
import requests
import pandas as pd
from datetime import datetime, timedelta

API_URL = "https://api.hyperliquid.xyz/info"

tickers = [
    'ATOM','REQ','CRV','MAVIA','SAGA','NEAR','MORPHO','MANTA','MOVE','XAI',
    'ETC','DOGE','SOPH','CELO','MAV','POPCAT','SCR','COMP','GMT','SOL','IMX',
    'JUP','RUNE','LAUNCHCOIN','UMA','TRB','USTC','AIXBT','IOTA','VIRTUAL',
    'ALGO','GMX','ANIME','BCH','BIO','BSV','NXPC','MOODENG','TNSR','HBAR',
    'SNX','ZEREBRO','HYPER','SAND','BERA','PURR','GAS','LDO','ONDO','DYDX',
    'FTT','TON','EIGEN','LTC','BLAST','AI16Z','OMNI','AAVE','OGN','SUI',
    'MEME','FXS','NEIROETH','NIL','CFX','ME','XRP','TIA','BNB','NOT','IP',
    'OM','TAO','OP','CAKE','AVAX','kPEPE','GALA','MNT','BOME','SUPER','SEI',
    'VINE','KAS','BABY','STX','S','FARTCOIN','STG','RENDER','ENA','LINK',
    'ARB','ARK','BIGTIME','BTC','ETH','RSR','kDOGS','BRETT','BANANA','XLM',
    'INJ','ENS','AR','DOT','SPX','ETHFI','PAXG','kLUNC','GOAT','kSHIB','FIL',
    'MEW','STRK','TRX','ZK','KAITO','PENGU','kBONK','VVV','ORDI','INIT','APT',
    'REZ','LAYER','ZEN','SUSHI','kFLOKI','ADA','kNEIRO','PEOPLE','ZORA',
    'PENDLE','APE','HYPE','FET','CHILLGUY','MELANIA','GRIFFAIN','PNUT','DOOD',
    'WIF','ACE','ZETA','TRUMP','NEO','JTO','YGG','ZRO','PROMPT','WLD','W',
    'MERL','BLUR','UNI','DYM','MINA','MKR','POLYX','POL','IO','TURBO','PYTH',
    'USUAL','GRASS','ALT','HMSTR','WCT','SYRUP','RESOLV','PROVE','YZY','WLFI',
    'TST','PUMP','LINEA','SKY','ASTER','0G','STBL','AVNT','XPL','ZEC','ICP'
]

def fetch_candles(coin, start_ms, end_ms, interval="1h"):
    payload = {
        "type": "candleSnapshot",
        "req": {
            "coin": coin,
            "interval": interval,
            "startTime": start_ms,
            "endTime": end_ms
        }
    }
    r = requests.post(API_URL, json=payload)
    try:
        r.raise_for_status()
    except:
        return pd.DataFrame()

    data = r.json()
    if not isinstance(data, list) or len(data) == 0:
        return pd.DataFrame()

    return pd.DataFrame(data)

# Last 30 days
now = datetime.utcnow()
start = datetime(2025, 6, 5)
start_ms = int(start.timestamp() * 1000)
end_ms = int(now.timestamp() * 1000)

rows = []

for ticker in tickers:
    print(f"Downloading {ticker}...")
    df = fetch_candles(ticker, start_ms, end_ms)

    if df.empty:
        print(f"⚠️ No data for {ticker} — skipped")
        continue

    # convert t → real datetime
    df["time"] = pd.to_datetime(df["t"], unit="ms")

    # keep only the opening candle at 18:00
    df_18 = df[df["time"].dt.hour == 18]

    for _, row in df_18.iterrows():
        rows.append({
            "perp": ticker,
            "time": row["time"],
            "close": row["c"]
        })

# Build final CSV
final_df = pd.DataFrame(rows)
final_df = final_df.sort_values(["perp", "time"])
final_df.to_csv("all_perps_18h_closing_prices.csv", index=False)

print("\nSaved → all_perps_18h_closing_prices.csv")


⚠️ No data for ANIME — skipped
⚠️ No data for BCH — skipped
⚠️ No data for BIO — skipped
⚠️ No data for BSV — skipped
⚠️ No data for TNSR — skipped
⚠️ No data for HBAR — skipped
⚠️ No data for ZEREBRO — skipped
⚠️ No data for BERA — skipped
⚠️ No data for PURR — skipped
⚠️ No data for GAS — skipped
⚠️ No data for LDO — skipped
⚠️ No data for DYDX — skipped
⚠️ No data for FTT — skipped
⚠️ No data for TON — skipped
⚠️ No data for EIGEN — skipped
⚠️ No data for LTC — skipped
⚠️ No data for BLAST — skipped
⚠️ No data for AI16Z — skipped
⚠️ No data for OMNI — skipped
⚠️ No data for AAVE — skipped
⚠️ No data for OGN — skipped
⚠️ No data for ARK — skipped
⚠️ No data for ETH — skipped
⚠️ No data for RSR — skipped
⚠️ No data for kDOGS — skipped
⚠️ No data for BANANA — skipped
⚠️ No data for XLM — skipped
⚠️ No data for ENS — skipped
⚠️ No data for AR — skipped
⚠️ No data for DOT — skipped
⚠️ No data for SPX — skipped
⚠️ No data for ETHFI — skipped
⚠️ No data for APE — skipped
⚠️ No data for HYP